# Generics

Welcome! Generics let you write code that works with many types without duplicating logic. Think of them as "type parameters" — you write the function or struct once, and the compiler generates specialized versions for each concrete type you use. Rust's generics are **zero-cost**: no runtime overhead.

## What Are Generics and Why Do They Matter?

Without generics, you'd need separate functions for `largest_i32`, `largest_f64`, `largest_char`, etc. With generics, one `largest<T>` handles all of them. The same idea applies to structs, enums, and impl blocks.

## Generic Functions

Declare type parameters in angle brackets after the function name. The parameter `T` can be used in the signature and body.

In [ ]:
fn largest<T: PartialOrd>(list: &[T]) -> Option<&T> {
    let mut largest = list.get(0)?;
    for item in list.iter() {
        if item > largest {
            largest = item;
        }
    }
    Some(largest)
}

let nums = [3, 1, 4, 1, 5];
let chars = ['x', 'a', 'm'];
println!("largest nums: {:?}", largest(&nums));
println!("largest chars: {:?}", largest(&chars));

## Generic Structs

Structs can have type parameters too. Each field can use a different generic type.

In [ ]:
struct Point<T> {
    x: T,
    y: T,
}

let int_point = Point { x: 1, y: 2 };
let float_point = Point { x: 1.0, y: 2.0 };
println!("int: ({}, {}), float: ({}, {})",
    int_point.x, int_point.y, float_point.x, float_point.y);

In [ ]:
// Multiple type parameters
struct Pair<T, U> {
    first: T,
    second: U,
}

let p = Pair { first: "hello", second: 42 };
println!("Pair: {} and {}", p.first, p.second);

## Generic Enums: Option&lt;T&gt; and Result&lt;T, E&gt;

Rust's standard library uses generics everywhere. `Option<T>` represents "maybe a value"; `Result<T, E>` represents "success with T or error with E".

In [ ]:
// Option<T> = Some(T) | None
let some_int: Option<i32> = Some(42);
let none_val: Option<i32> = None;

// Result<T, E> = Ok(T) | Err(E)
let ok_result: Result<i32, &str> = Ok(100);
let err_result: Result<i32, &str> = Err("something went wrong");

println!("Option: {:?}, {:?}", some_int, none_val);
println!("Result: {:?}, {:?}", ok_result, err_result);

## Generic impl Blocks

You can implement methods for generic structs. The impl block declares the same type parameters.

In [ ]:
impl<T> Point<T> {
    fn x(&self) -> &T {
        &self.x
    }
}

// impl only for Point<f32>
impl Point<f32> {
    fn distance_from_origin(&self) -> f32 {
        (self.x * self.x + self.y * self.y).sqrt()
    }
}

let p = Point { x: 3.0, y: 4.0 };
println!("distance: {}", p.distance_from_origin());

## Trait Bounds on Generics

Often `T` must support certain operations. Use trait bounds: `T: Trait`. This tells the compiler "T must implement Trait".

In [ ]:
use std::fmt::Display;

fn print_pair<T: Display, U: Display>(first: T, second: U) {
    println!("{} and {}", first, second);
}

print_pair(1, "two");
print_pair(3.14, 42);

## Multiple Bounds with +

When a type must implement multiple traits, use `+` to combine them.

In [ ]:
fn compare_and_print<T: PartialOrd + Display>(a: T, b: T) {
    if a >= b {
        println!("{} >= {}", a, b);
    } else {
        println!("{} < {}", a, b);
    }
}

compare_and_print(10, 20);

## Where Clauses

When bounds get long, move them to a `where` clause for readability.

In [ ]:
fn complex_generic<T, U>(t: T, u: U)
where
    T: Display + Clone,
    U: Display + PartialOrd,
{
    println!("t: {}, u: {}", t, u);
}

complex_generic("hello", 42);

## Generic Return Types

Functions can return generic types. The compiler infers the concrete type from how the result is used.

In [ ]:
fn first<T>(list: &[T]) -> Option<&T> {
    list.first()
}

let v = vec![1, 2, 3];
let f: Option<&i32> = first(&v);
println!("first: {:?}", f);

## Monomorphization — Zero-Cost Abstractions

Rust compiles generics by **monomorphization**: the compiler generates a separate copy of the code for each concrete type used. So `largest::<i32>` and `largest::<char>` become two distinct functions. No runtime dispatch, no performance penalty.

## Turbofish Syntax ::&lt;&gt;

When the compiler can't infer the type, use the **turbofish** to specify it explicitly: `function::<Type>(args)`.

In [ ]:
let v: Vec<i32> = vec![1, 2, 3];
let parsed: i32 = "42".parse().unwrap();
// parse() returns Result<T, E> — need to specify T:
let also_parsed = "99".parse::<i32>().unwrap();
println!("parsed: {}, also_parsed: {}", parsed, also_parsed);

## Const Generics

Rust also supports **const generics**: type-level constants. Useful for fixed-size arrays and similar patterns.

In [ ]:
struct ArrayWrapper<T, const N: usize> {
    data: [T; N],
}

let w: ArrayWrapper<i32, 3> = ArrayWrapper { data: [1, 2, 3] };
println!("ArrayWrapper<3>: {:?}", w.data);

## PhantomData (Brief Mention)

`PhantomData<T>` is a zero-sized type used to "mark" that a struct logically owns or uses `T` even if it doesn't store a value of type `T`. Useful for unsafe code and variance. We'll skip the details here — just know it exists.

## Standard Generic Types

You've already seen many: `Vec<T>`, `Option<T>`, `Result<T,E>`. Others include `Box<T>` (heap allocation), `Rc<T>` (reference counting), `RefCell<T>` (interior mutability).

In [ ]:
use std::rc::Rc;

let boxed: Box<i32> = Box::new(42);
let rc: Rc<String> = Rc::new(String::from("shared"));
println!("boxed: {}, rc: {}", boxed, rc);

## Common Mistakes Beginners Make

1. **Forgetting trait bounds** — e.g. using `>` on `T` without `T: PartialOrd`
2. **Over-specifying bounds** — only add bounds you actually need
3. **Confusing `impl Trait` and generics** — `fn f<T: Display>(x: T)` vs `fn f(x: impl Display)`
4. **Turbofish in wrong place** — it's `f::<T>(args)`, not `f(args::<T>)`

## Key Rules to Remember

- Generics = type parameters; write once, use with many types
- Trait bounds (`T: Trait`) restrict what `T` can do
- `where` clauses keep signatures readable
- Monomorphization = zero-cost, no runtime overhead
- Turbofish `::<T>` when the compiler needs a type hint
- `Option<T>`, `Result<T,E>`, `Vec<T>` are all generic